# 01. LunarLander-v2 PPO 학습
> **Google Colab T4** 에서 위에서 아래로 순서대로 셀을 실행하세요.

## 1. 패키지 설치

In [ ]:
!pip install -q stable-baselines3==2.3.2 gymnasium[box2d]==0.29.1 shimmy==1.3.0 tensorboard>=2.9.1

## 2. 환경 확인

In [ ]:
import gymnasium as gym
import numpy as np

env = gym.make('LunarLander-v2')
obs, _ = env.reset()
print('Observation shape:', obs.shape)
print('Action space:', env.action_space)
env.close()

## 3. PPO 학습

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import EvalCallback
import os

os.makedirs('/content/results/models', exist_ok=True)
os.makedirs('/content/results/plots', exist_ok=True)

vec_env = make_vec_env('LunarLander-v2', n_envs=4)

model = PPO(
    'MlpPolicy', vec_env,
    learning_rate=3e-4, n_steps=1024, batch_size=64,
    n_epochs=10, gamma=0.999, gae_lambda=0.98, ent_coef=0.01,
    verbose=1, device='auto',
)

eval_callback = EvalCallback(
    make_vec_env('LunarLander-v2', n_envs=1),
    best_model_save_path='/content/results/models/',
    log_path='/content/results/',
    eval_freq=5000, n_eval_episodes=10, deterministic=True, verbose=1,
)

model.learn(total_timesteps=300_000, callback=eval_callback)
model.save('/content/results/models/ppo_lunarlander')
print('학습 완료')

## 4. 학습 곡선 시각화

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

results = np.load('/content/results/evaluations.npz')
timesteps = results['timesteps']
mean_rewards = results['results'].mean(axis=1)

plt.figure(figsize=(10, 4))
plt.plot(timesteps, mean_rewards, marker='o', markersize=3)
plt.axhline(200, color='red', linestyle='--', label='Solved (200)')
plt.xlabel('Timesteps'); plt.ylabel('Mean Reward'); plt.title('LunarLander-v2 Learning Curve')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig('/content/results/plots/lunarlander_curve.png', dpi=150)
plt.show()

## 5. 에이전트 평가

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy
import gymnasium as gym

best_model = PPO.load('/content/results/models/best_model')
eval_env = gym.make('LunarLander-v2')
mean_reward, std_reward = evaluate_policy(best_model, eval_env, n_eval_episodes=20, deterministic=True)
print(f'Mean reward: {mean_reward:.2f} ± {std_reward:.2f}')
eval_env.close()

---
다음 → **02_trading_ppo.ipynb** 에서 동일한 PPO 구조를 커스텀 트레이딩 환경에 전이합니다.